In [1]:
from gen_catalyst_design.discrete_space_diffusion import ExponentialBetaScheduler, UniformTransitionsNoiser, AbsorbingStateNoiser, CosineScheduler
from gen_catalyst_design.stability import apply_inversion_symmetry
from gen_catalyst_design.utils import get_full_element_pool_no_saas
from gen_catalyst_design.discrete_space_diffusion.Dataset import get_dataloaders_from_atoms_list
from ase_ml_models.databases import get_atoms_list_from_db
from ase_ml_models.utilities import get_connectivity, plot_connectivity
from ase.db import connect
import torch
from ase.io import write
from ase.visualize import view
import os


In [2]:
miller_index = "100"
surface_type = "cluster"
num_copies = 6
element_pool = get_full_element_pool_no_saas()
element_pool_no_abs = element_pool.copy()
noiser_type = "Absorbing"
timesteps = torch.arange(0, 1001, 1) #[0, 250, 500, 750, 1000]

scheduler = CosineScheduler(
        #beta_max=5e-2, 
        #beta_min=1e-4,
        time_sample_method="stratified"
    )

if noiser_type == "Absorbing":
    element_pool = ["(X)"] + element_pool

if noiser_type == "Absorbing":
    noiser = AbsorbingStateNoiser(
                element_pool=element_pool
    )
if noiser_type == "Uniform":
    noiser = UniformTransitionsNoiser(
        element_pool=element_pool
    )

noiser.pre_compute_accum_q_matrices(scheduler=scheduler)

ase_db = connect(f"../../databases/{surface_type}_templates/{miller_index}_templates.db")
template_atoms_list = get_atoms_list_from_db(ase_db)
template_atoms = template_atoms_list[0]
atoms_list = []
for element in element_pool_no_abs[:num_copies]:
    atoms = template_atoms.copy()
    atoms.symbols = [element for _ in range(len(template_atoms))]
    atoms_list.append(atoms)

cell = template_atoms.get_cell()

train_loader, val_loader = get_dataloaders_from_atoms_list(
    atoms_list=atoms_list,
    element_pool=element_pool,
    batch_size=num_copies,
    train_val_split=0,
    do_initial_shuffling=False,
    do_train_shuffling=False,
)

tot_atoms_list_dict = {}
for batch in train_loader:
    batch_copy = batch.clone()
    for timestep in timesteps:
        noiser.noise_batch_xtm1_xt(
            batch=batch_copy,
            time_batch=timestep*torch.ones(size=(batch_copy.num_nodes,), dtype=torch.long),
            scheduler=scheduler
        )
        for sample_idx in range(num_copies):
            graph = batch_copy.get_example(sample_idx)
            atoms = graph.to_atoms(element_pool)
            if sample_idx in tot_atoms_list_dict:
                tot_atoms_list_dict[sample_idx].append(atoms)
            else:
                tot_atoms_list_dict[sample_idx] = [atoms]

In [3]:
for element_idx in tot_atoms_list_dict:
    write(f"{noiser_type}/noise_{element_idx}.traj", tot_atoms_list_dict[element_idx])

In [ ]:
timesteps = [0, 250, 500, 750, 1000]
tot_atoms_dict = {}
for timestep in timesteps:
    noised_samples = []
    for batch in train_loader:
        batch_copy = batch.clone()
        noiser.noise_batch_x0_xt(batch=batch_copy, time_batch=timestep*torch.ones(size=(batch_copy.num_nodes,), dtype=torch.long))
        for sample_idx in range(num_copies):
            graph = batch_copy.get_example(sample_idx)
            atoms = graph.to_atoms(element_pool)
            noised_samples.append(atoms)
    tot_atoms_dict[timestep] = noised_samples

    